Ejecuta la siguiente celda e introduce la contraseña facilitada en el curso para desbloquear el cuaderno.


In [ ]:
%pip install -q git+https://github.com/dcrespons14/cam-estrategia-aula.git
from cam_estrategia import desbloquear_cuaderno
desbloquear_cuaderno()


# Solución al ejercicio de 3.7: Pepe Martí

En la clase 3.7 estudiamos cuándo debía activar el Attack Mode Pascal Wehrlein, que salía desde la pole. El ejercicio consistía en repetir el mismo análisis desde el punto de vista de Pepe Martí, que sale decimotercero.

¿Qué diferencias hay? ¿Por qué ocurren estas diferencias? ¿Cuál es la mejor estrategia?

## Inicialización

Utilizaremos exactamente la misma configuración que en la clase 3.7.

In [ ]:
# Librerías propias (ver src/race_models/fe)
from race_models.fe.config import RaceConfig
from race_models.fe.monte_carlo import run_monte_carlo
from race_models.fe.analysis.mc_plots import plot_monte_carlo_exp, plot_monte_carlo_dist
from race_models.fe.analysis.mc_tables import monte_carlo_dataframe
from race_models.fe.analysis.race_plots import plot_race

# Librerías de Python
import pandas as pd
import numpy as np
from IPython.display import display

In [ ]:
race_config = RaceConfig()
activation_plans = [(lap,) for lap in range(3, race_config.total_laps + 1)]  # Activaciones entre la vuelta 3 y la 25

## Monte Carlo - Simulación

Lo único que cambia respecto a la Parte 1 es la posición de salida.

In [ ]:
result = run_monte_carlo(
    race_config=race_config,
    starting_position=13,  # Pepe Martí sale decimotercero
    activation_plans=activation_plans,
    simulations=2500,
    sc=True,
    seed=42
)

print(f"Carreras simuladas: {len(result.strategies) * len(result.scenario_seeds):,}")

## Valores esperados y desviaciones estándar

La tabla está ordenada por puntos medios.

In [ ]:
df = monte_carlo_dataframe(result)
with pd.option_context('display.max_rows', None, 'display.float_format', '{:.3f}'.format):
    display(df)

In [ ]:
plot_monte_carlo_exp(result);

### Primera diferencia: puntos y posición no dicen lo mismo

Con Wehrlein, la estrategia con más puntos medios era también la de mejor posición media. Con Martí no ocurre: la estrategia que más puntos da no es la que le deja mejor posicionado de media.

## Distribuciones

Comparamos dos estrategias: activar pronto (vuelta 6, la de más puntos medios) y activar tarde (vuelta 17, la de mejor posición media).

In [ ]:
plot_monte_carlo_dist(result, activation_lap=[6, 17]);

### Segunda diferencia: fuera de los puntos, hay que arriesgar

Desde la decimotercera posición, acabar 12º o 15º da exactamente igual: 0 puntos. Lo único que importa es la probabilidad de entrar en el top 10.

Activar pronto da una distribución más ancha (mayor desviación estándar): más carreras malas, pero también más carreras en las que Martí entra en los puntos. Activar tarde da resultados más concentrados alrededor de la posición de salida, es decir, fuera de los puntos.

Con Wehrlein ocurría lo contrario: desde la pole queremos proteger la posición, así que buscamos la estrategia menos arriesgada.

## Ver una carrera individual

Una carrera en la que Martí activa en la vuelta 6.

In [ ]:
activation_lap = 6
race_result = result.get_race(np.random.randint(0, 1000), strategy_index=activation_lap - 3)
plot_race(race_result, title=f"Monte Carlo - Martí, activación L{activation_lap}");